# Data 207 Modeling - Nate del Rosario

This notebook is solely for modeling. All EDA and necessary plots can be found in **'src/eda.ipynb'**

In [1]:
# import sys
# !{sys.executable} -m pip install xgboost

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMClassifier
from imblearn.over_sampling import RandomOverSampler

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, make_scorer, recall_score,
    roc_curve, f1_score, accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay
)

scoring = make_scorer(recall_score, pos_label=1)

In [3]:
# load train data
X_train_raw = pd.read_parquet("../data/model_raw/X_train.parquet")
y_train_raw = pd.read_parquet("../data/model_raw/Y_train.parquet")
y_train_raw = np.where(y_train_raw['depression_severity'] > 0, 1, 0) # create binary labels

# upsample train data - do what you need to do
ros = RandomOverSampler(sampling_strategy='not majority', random_state=7)
X_train_balanced, y_train_balanced = ros.fit_resample(X_train_raw, y_train_raw)

# convert X and Y to np array
X_train = np.array(X_train_balanced)
y_train = np.array(y_train_balanced)

# load validation data
X_val = pd.read_parquet("../data/model_raw/X_val.parquet")
y_val = pd.read_parquet("../data/model_raw/Y_val.parquet")['depression_severity']
X_val = np.array(X_val)
y_val = np.array(np.where(y_val > 0, 1, 0)) # np array of binary labels

# load test data
X_test = pd.read_parquet("../data/model_raw/X_test.parquet")
y_test = pd.read_parquet("../data/model_raw/Y_test.parquet")['depression_severity']
X_test = np.array(X_test)
y_test = np.array(np.where(y_test > 0, 1, 0)) # np array of binary labels

In [4]:
print(X_train.shape)

(4514, 146)


### Handling Class Imbalance

As a baseline, we stick upsampling as EDA and experiments suggested it provides a better generalizing model

In [5]:
classes = np.unique(y_train)
classes

array([0, 1])

### Feature Selection on Numeric Columns Only

### Feature Importance via XGBoost

We use XGBoost to see how valuable each feature is in constructing its decision trees

In [6]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [7]:
# selection based on importance
selector = SelectFromModel(model, threshold="median", prefit=True)
selected_features = pd.DataFrame(X_train).columns[selector.get_support()]
selected_features

Index([  1,   4,   5,   7,   8,   9,  13,  14,  16,  18,  21,  22,  25,  27,
        31,  32,  33,  34,  35,  36,  37,  49,  60,  75,  80,  81,  82,  83,
        85,  86,  87,  88,  91,  92,  94,  96,  97,  98,  99, 100, 101, 102,
       103, 104, 106, 107, 108, 109, 110, 112, 113, 115, 116, 117, 118, 120,
       121, 122, 123, 124, 125, 126, 127, 128, 130, 131, 132, 133, 134, 135,
       143, 144, 145],
      dtype='int64')

In [8]:
feature_names = X_train_raw.columns.tolist()

importance_df = pd.DataFrame({
    'feature'   : feature_names,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

# apply the same median threshold filter
threshold = importance_df['importance'].median()
selected_df = importance_df[importance_df['importance'] >= threshold]

print(f'Selected {len(selected_df)} / {len(feature_names)} features (threshold={threshold:.2f})')
selected_df.head(10)

Selected 73 / 146 features (threshold=0.01)


,feature,importance
32,trouble_sleeping_Yes,0.096664
35,sleepy_frequency_Often,0.031042
34,sleepy_frequency_Never,0.025950
33,sleepy_frequency_Almost always,0.021953
1,max_education_College graduate or above,0.020194
134,days_cigarette_smoked,0.019697
9,marital_status_Married,0.018736
135,cigarette_count,0.018606
36,sleepy_frequency_Rarely,0.016669
16,gender_male,0.016376


### Feature Importance via Performance Permutation 

This works by telling us much our model's performance decreases when a single feature’s values are randomly shuffled. For example, if we randomly shuffle the values in 'days_cigarette_smoke' row-wise and accuraacy significanty drops, then that feature is important.

In [9]:
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=1
)

importance_df = pd.DataFrame({
    "feature": X_train_raw.columns.tolist(),
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

importance_df.head(20)

,feature,importance
32,trouble_sleeping_Yes,0.015187
36,sleepy_frequency_Rarely,0.012623
35,sleepy_frequency_Often,0.011045
82,2_year_interview_weight,0.009961
80,income_poverty_ratio,0.008087
33,sleepy_frequency_Almost always,0.007495
97,basophil_perc,0.006903
118,total_cholesterol_mg_dl,0.006509
106,mean_platelet_vol,0.005128
125,vitamin_d3,0.004734
